In [ ]:
# THE GRAPH SPRINT: END-TO-END DOCUMENT SUMMARISER

"""
This is the final day of week 7, and it is a pure intergration sprint. No new concepts today.
Instead, I combine everything built across days 14-19 (LangGraph state machines, cycles, checkpointing,
multi-agent Researcher/Writer, and error-resilient fallback) into one capstone pipeline: a Document Summariser that works
through 5 distinct sections of Apple FY2024 10-K, researhces and writes each one using the full multi-agent
workflow, survives API failures without crashing, and compiles everything into one cohesive, checkpointed report.

Deep coding

Section Loop: Extending the Day 18 multi-agent graph so it processes a list of 5 topics sequentially
instead of just one - a new advance_section_node moves to the next topic once the Writer is satisfied with the current one
and only proceeds to compile_node once all 5 are done.
Error Resilience Inside the Loop: Wrap the Researcher's LLM call with the same try/except -> Ollama fallback logic from day 19, so a Groq
failure on section 3 of 5 does not kill the whole report - it degrades gracefully and keeps going
Checkpointed MUlti-Section Run: Compile the graph with sqlitesaver from day 16, so if the process is interrupted after 2 of 5 sections, you
can resume from exactly where it stopped rather than re-researching everything.

"""

"\nThis is the final day of week 7, and it is a pure intergration sprint. No new conceptstoday.\nInstead, I combine everything built across days 14-19 (LangGraph state machines, cycles, checkpointing,\nmulti-agent Researcher/Writer, and error-resilient fallback) into one capstone pipeline: a Document Summariser that works\nthrough 5 distinct sections of Apple FY2024 10-K, researhces and writes each one using the full multi-agent\nworkflow, survives API failures without crashing, and compiles everything into one cohesive, checkpointed report.\n\nDeep coding\n\nSection Loop: Extending the Day 18 multi-agent graph so it processes a list of 5 topics sequentially\ninstead of just one - a new advance_section_node moves to the next topic once the Writer is satisfied with the current one\nand only proceeds to compile_node once all 5 are done.\nError Resilience Inside the Loop: Wrap the Researcher's LLM call with the same try/except -> Ollama fallback logic from day 19, so a Groq\nfailure on se

In [2]:


# Importing necessary libraries


import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional, TypedDict, Literal
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from langchain_community.tools.tavily_search import TavilySearchResults

Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)


load_dotenv(find_dotenv())

client = Groq()
webSearch = TavilySearchResults(max_results = 3)


# LOADING EXISTING VECTOR STORE

# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')


# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

def format_chunks(docs: list) -> str:
    return "\n\n---\n\n".join([doc.node.get_content() for doc in docs])
        


# CRAG -> Retrieval quality gate

def corrective_retrieve(query: str) -> tuple[str, str]:
    """
    Returns (context_text, source) where source is 'local' or 'web'.
    Applies CRAG logic: low confidence -> discard local, use web fallback.
    """

    docs, top_score = retrieve_with_confidence(query)

    if top_score < RELEVANCE_THRESHOLD or not docs:
        print("🤥 CRAG: Low retrieval confidence - falling back to web search")
        webResults = webSearch.invoke(query)
        context = "\n\n".join([r["content"] for r in webResults])
        return context, "web"
    
    else:
        print("👍 CRAG: Retrieval confidence acceptable - using local docs")
        return format_chunks(docs), "local"
    

GENERATOR_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the user's question using ONLY the context provided.
If the context does not contain eough information to answer, say exactly:
'I cannot find sufficient information in the provided context.' 
Be specific - include numbners, percentages, and fiscal year references where available.

"""

REVIEWER_SYSTEM_PROMPT = """You are a strict factual reviewer for a financial RAG system.
You will receive a question, the source context, and a generated answer.

Your job is to check:
1. Does the answer contain any claims NOT supported by the context? (hallucination)
2. Does the answer actually address the question asked?
3. Are numbers, percentages, and figures accurate relative to the context?

Respond in EXACTLY this format:
Verdict: <PASS or FAIL>
Reason: <one sentence explaining your verdict>

PASS meaans the answer is faithful to the context and addresses the question,
FAIL means the answer contains unsupported claims, wrong figuress, or avoids the question.


"""

ROUTER_SYSTEM_PROMPT = """You are a query complexity classifier for a financial RAG system.

Classify the user's question as one of:
- SIMPLE: a single factual lookup requiring one retrieval and one answer
(e.g. "What was Apple's net income in FY2024?")
- COMPLEX: requires multiple steps, comparisons, calculations, or chaining
(e.g. "Compare iPhone revenue across FY2023 and FY2024 and calculate the growth rate")
- UNKNOWN: cannot be answered from a financial document at all 
(e.g. "What is the weather in Cupertino today?")

Respond in EXACTLY this format:
Classification: <SIMPLE, COMPLEX, or UNKNOWN>
Reason: <one sentence>
"""

def generate_answer(question: str, context: str, critique: str = "") -> str:
    critiqueBlock = ""
    if critique:
        critiqueBlock = f"\n\nPrevious answer was rejected for this reason: {critique}\nPlease rewrite addressing this critique."

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = [
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
                f"{critiqueBlock}"
            )}
        ],

        temperature= 0,
    )

    return response.choices[0].message.content


def review_answer(question: str, context: str, answer: str) -> tuple[str, str]:
    """Returns (verdict, reason) where verdict is a PASS or FAIL."""
    response = client.chat.completions.create(
        model = 'openai/gpt-oss-120b',
        messages = [
            {"role": "system", "content": REVIEWER_SYSTEM_PROMPT},
            {"role": "user", "content":(
                f"Question: {question}\n\n"
                f"Source Context:\n{context}\n\n"
                f"Generated Answer:\n{answer}"

            )}
        ],
        temperature= 0
    )

    raw = response.choices[0].message.content
    verdict_match = re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match =re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f"😮‍💨 Reviewer verdict: {verdict} - {reason}")
    return verdict, reason

# NAIVE RAG PATH 

NAIVE_RAG_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the question uning ONLY THE context provided.
Be specific - include exact figures, percentages, and fiscal year references.
If the context does not contain the answer, say so directly.

"""

def run_naive_rag(question: str) -> dict:
    print("\n⚡ Path: NAIVE RAG")
    started_at = datetime.now().isoformat()

    # single retrieval
    docs, score = retrieve_with_confidence(question)
    context = "\n\n --- \n\n".join([doc.node.get_content() for doc in docs])

    # Single generation - no reviewer, no rewrite
    answer = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }],
        system = NAIVE_RAG_SYSTEM_PROMPT
    )

    print(f"Answer: {answer}")
    return {
        "path": "naive_rag",
        "question": question,
        "answer": answer,
        "retrieval_score": score,
        "started_at": started_at,
        "finished_at": datetime.now().isoformat()
    }


# THE FULL SELF CORRECTING CRAG PIPELINE

def run_crag_pipeline(question: str, max_rewrites: int = 2) ->dict:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'='*60}")

    startedAt = datetime.now().isoformat()

    # STEP 1: CRAG retrieval with confidence gate
    context, source = corrective_retrieve(question)

    # STEP 2: Generate Initial Answer
    print("\n 🦾 Generating initial answer...")
    answer = generate_answer(question, context= context)
    print(f"Answer: {answer}\n")

    # SECONDARY GATE to catch false-positive vector scores
    if answer and "I cannot find sufficient information" in answer and source == "local":
        print("🤥 CRAG: Local docs failed to answer despite high vector score. Forcing web fallback...")
        webResults = webSearch.invoke(question)
        context = "\n\n".join([r["content"] for r in webResults])
        source = "web"

        print("👍Generating answer from web context...")
        answer = generate_answer(question, context= context)
        print(f"Web Fallback Answer: {answer}\n")

    # STEP 3: Reviewer Loop
    attempts = 0
    verdict = 'FAIL'
    critique = ""
    history = []

    while verdict == 'FAIL' and attempts < max_rewrites:
        verdict, critique = review_answer(question= question, context= context, answer= answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

        if verdict == 'FAIL':
            attempts += 1
            if attempts < max_rewrites:
                print(f"\n🔁 Rewriting (attempt {attempts})...")
                answer = generate_answer(question, context, critique)
                print(f"Rewritten Answer: {answer}\n")
            else:
                print("🤥 Max rewrites reached - returning best attempt with warning")

    # final evrdict check if we exited the loop with PASS 
    if verdict != 'FAIL':
        verdict, critique = review_answer(question, context, answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

    result = {
        "question": question,
        "retrieval_source": source,
        "final_answer": answer,
        "fianl_verdict": verdict,
        "rewrite_attempts": attempts,
        "review_history": history,
        "started_at":startedAt,
        "finished_at": datetime.now().isoformat()
    }


    print(f"\n{'='*60}")
    print(f"👍 Final Answer ({verdict} after {attempts} rewrite(s)):")
    print(answer)
    print(f"{'='*60}\n")

    filename = f"traces/day11_crag_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent= 2)
    print(f"📀 Saved to {filename}")

    return result


def classify_query(question: str) -> tuple[str, str]:
    raw = tracked_llm_call(
        messages = [{"role": "user", "content": f"Question: {question}"}],
        system= ROUTER_SYSTEM_PROMPT
    )
    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not Parse reason."

    print(f"🦾 Router: {classification} - {reason}")
    return classification, reason

# MULTI-PATH ROUTER

def run_router(question: str) -> dict:
    global usage_tracker
    usage_tracker = TokenUsage() # reset for each question

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    classification, reason = classify_query(question)

    if classification== 'SIMPLE': 
        result = run_naive_rag(question= question)

    elif classification == "COMPLEX":
        print("\n🤖 Path: AGENTIC RAG (CRAG + Reviewer)")
        result = run_crag_pipeline(question= question)
        result['path'] = "agentic_rag"
    
    else: # UNKNOWN
        print("\n🤥 Path: UNKNOWN - cannot answer from financial documents")
        result = {
            "path": "unknown",
            "question": question,
            "answer": "This question cannot be answered using the Apple 10-K document.",
            "started_at": datetime.now().isoformat(),
            "finished_at": datetime.now().isoformat()
        }
    
    usage_tracker.report()

    result["Classification"] = classification
    result["classification_reason"] = reason
    result["token_usage"] = {
        "prompt_tokens": usage_tracker.prompt_tokens,
        "completion_tokens": usage_tracker.completion_tokens,
        "total_calls": usage_tracker.total_calls,
        "estimated_cost_usd": usage_tracker.cost_estimate()
    }

    filename = f"traces/day12_{classification.lower()}_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(result, f, indent = 2)
    print(f"\n📀 Saved to {filename}")

    return result



c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rodne\AppData\Local\Temp\ipykernel_15080\2096247782.py:30: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  webSearch = TavilySearchResults(max_results = 3)


🤖🛩️ Vector Store connection established ⚡


In [3]:
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2:1b"

# The Ollama fallback caller
import requests
from groq import APITimeoutError, RateLimitError, APIConnectionError

def call_ollama(messages: list, system: str = "") -> str:

    """
    Calls a locally running Ollama instance.
    Uses the same message format as Groq so it is a drop-in replacement.
    """
    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": OLLAMA_MODEL,
                "messages": full_messages,
                "stream": False,
                "options": {"temperature": 0}
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["message"]["content"]

    except requests.exceptions.ConnectionError:
        return (
            "FALLBACK_UNAVAILABLE: Ollama is not running locally. "
            "Start Ollama with 'ollama serve' in your terminal."
        )
    except requests.exceptions.Timeout:
        return (
            "FALLBACK_UNAVAILABLE: Ollama timed out. "
            "The model may be loading — try again in 30 seconds."
        )
    except Exception as e:
        return f"FALLBACK_UNAVAILABLE: Unexpected Ollama error — {str(e)}"


def check_ollama_available() -> bool:
    """Quick health check before attempting fallback."""
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = [m["name"] for m in response.json().get("models", [])]
        available = any(OLLAMA_MODEL in m for m in models)
        if available:
            print(f"✅ Ollama available — {OLLAMA_MODEL} is loaded")
        else:
            print(f"⚠️  Ollama running but {OLLAMA_MODEL} not found")
            print(f"   Available models: {models}")
            print(f"   Run: ollama pull {OLLAMA_MODEL}")
        return available
    except Exception:
        print(f"❌ Ollama not reachable at {OLLAMA_BASE_URL}")
        return False

In [4]:
# THE MULTI-SECTION STATE SCHEMA

import uuid
import sqlite3
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver
from typing import TypedDict, Optional, Literal
from datetime import datetime

DB_PATH = "checkpoints/rag_checkpoints.db"
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("traces", exist_ok=True)

class DocSummariserState(TypedDict):
    # The 5 sections to work through
    sections: list[str]
    section_index: int

    # Per-section working state (reset at the start of each section)
    current_topic: str
    research_packets: list[dict]
    researcher_attempts: int
    max_researcher_attempts: int
    follow_up_query: Optional[str]
    needs_more_research: bool
    writer_attempts: int

    # Accumulates across ALL sections
    report_sections: list[str]

    # Error resilience
    api_error: bool
    api_error_type: Optional[str]
    using_fallback_llm: bool
    error_log: list[dict]

    # Final output
    final_report: Optional[str]
    finished_at: Optional[str]

In [15]:
# THE ERROR-RESILIENT RESEARCHER NODE
# This is Day 18's researcher_node with Day 19's try/except -> Ollama fallback wrapped directly around the LLM call.

RESEARCHER_SYSTEM_PROMPT = """You are a specialist financial researcher working on
an Apple Inc. FY2024 10-K analysis.

You will receive a research query. Identify the 3 most important facts or figures
that answer it, and note your confidence in each.

Respond in EXACTLY this format:

EVIDENCE 1:
Fact: <the specific fact or figure>
Confidence: <HIGH, MEDIUM, or LOW>

EVIDENCE 2:
Fact: <the specific fact or figure>
Confidence: <HIGH, MEDIUM, or LOW>

EVIDENCE 3:
Fact: <the specific fact or figure>
Confidence: <HIGH, MEDIUM, or LOW>

COVERAGE: <COMPLETE, PARTIAL, or INSUFFICIENT>
COVERAGE_REASON: <one sentence>
"""

def researcher_node(state: DocSummariserState) -> dict:
    query = state.get("follow_up_query") or state["current_topic"]
    attempts = state.get("researcher_attempts", 0) + 1

    print(f"\n[NODE: researcher] Section {state['section_index']+1}/5 — Attempt {attempts}")
    print(f"  → Query: {query[:80]}")

    # Retrieve from Apple 10-K vector store
    retriever = index.as_retriever(similarity_top_k = 4)
    docs = retriever.retrieve(query)

    # Calculate the score manually from the first document (if it exists)
    score = docs[0].score if docs and docs[0].score is not None else 0.0

    if not docs:
        context = "No relevant content found in the Apple FY2024 10-K for this query."
    else:
        context = "\n\n---\n\n".join([doc.node.get_content() for doc in docs])

    messages = [{
        "role": "user",
        "content": f"Research Query: {query}\n\nRetrieved Context:\n{context}"
    }]

    error_occurred = False
    error_type = None
    using_fallback = state.get("using_fallback_llm", False)

    try:
        raw = tracked_llm_call(messages=messages, system=RESEARCHER_SYSTEM_PROMPT)
    except (APITimeoutError, RateLimitError, APIConnectionError) as e:
        error_occurred = True
        error_type = type(e).__name__
        print(f"  ⚠️  Groq error ({error_type}) — falling back to Ollama")
        raw = call_ollama(messages=messages, system=RESEARCHER_SYSTEM_PROMPT)
        using_fallback = True

    evidence_blocks = re.findall(
        r"EVIDENCE \d+:\nFact: (.+?)\nConfidence: (HIGH|MEDIUM|LOW)", raw, re.DOTALL
    )
    coverage_match = re.search(r"COVERAGE: (COMPLETE|PARTIAL|INSUFFICIENT)", raw)
    coverage_reason_match = re.search(r"COVERAGE_REASON: (.+)", raw)

    evidence_list = [{"fact": f.strip(), "confidence": c} for f, c in evidence_blocks]
    coverage = coverage_match.group(1) if coverage_match else "PARTIAL"
    coverage_reason = (
        coverage_reason_match.group(1).strip() if coverage_reason_match
        else "Could not assess coverage."
    )

    packet = {
        "topic": state["current_topic"],
        "query": query,
        "evidence": evidence_list,
        "coverage": coverage,
        "coverage_reason": coverage_reason,
        "used_fallback": using_fallback,
    }

    print(f"  → Coverage: {coverage} — {coverage_reason}")

    updated_log = state.get("error_log", [])
    if error_occurred:
        updated_log = updated_log + [{
            "timestamp": datetime.now().isoformat(),
            "section": state["current_topic"],
            "error_type": error_type,
            "fallback_used": "ollama"
        }]

    return {
        "research_packets": state.get("research_packets", []) + [packet],
        "researcher_attempts": attempts,
        "follow_up_query": None,
        "needs_more_research": False,
        "using_fallback_llm": using_fallback,
        "error_log": updated_log,
        "api_error": error_occurred,
        "api_error_type": error_type,
    }

In [16]:
# WRITER NODE (THIS IS USED DIRECTLY FROM DAY 18, NOTHING IS CHANGED)

WRITER_SYSTEM_PROMPT = """You are a specialist financial report writer.
You will receive a topic and structured research evidence from an Apple FY2024 10-K analysis.

Write ONE polished report section:
- Clear heading (## Topic Name)
- 2-3 paragraphs grounded in the evidence
- A small data table if the evidence contains multiple figures
- End with "Key Takeaway:"

Then add:
NEEDS_MORE_RESEARCH: <YES or NO>
FOLLOW_UP_QUERY: <specific query, or NONE>
REASON: <one sentence>
"""

def writer_node(state: DocSummariserState) -> dict:
    attempts = state.get("writer_attempts", 0) + 1
    print(f"\n[NODE: writer] Section {state['section_index']+1}/5 — Attempt {attempts}")

    section_packets = [
        p for p in state.get("research_packets", [])
        if p["topic"] == state["current_topic"]
    ]
    all_evidence = [
        f"[{ev['confidence']} confidence] {ev['fact']}"
        for packet in section_packets for ev in packet.get("evidence", [])
    ]
    evidence_text = "\n".join([f"• {e}" for e in all_evidence])

    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"Topic: {state['current_topic']}\n\nResearch Evidence:\n{evidence_text}"
        }],
        system=WRITER_SYSTEM_PROMPT
    )

    parts = re.split(r"\nNEEDS_MORE_RESEARCH:", raw, maxsplit=1)
    report_section = parts[0].strip()
    needs_more, follow_up = False, None

    if len(parts) > 1:
        meta = parts[1]
        needs_match = re.search(r"^\s*(YES|NO)", meta)
        follow_up_match = re.search(r"FOLLOW_UP_QUERY:\s*(.+)", meta)
        needs_more = (needs_match.group(1) if needs_match else "NO") == "YES"
        follow_up_raw = follow_up_match.group(1).strip() if follow_up_match else "NONE"
        follow_up = None if follow_up_raw == "NONE" else follow_up_raw

    print(f"  → Section written ({len(report_section)} chars) | needs_more: {needs_more}")

    return {
        "report_sections": state.get("report_sections", []) + [report_section],
        "needs_more_research": needs_more,
        "follow_up_query": follow_up,
        "writer_attempts": attempts,
    }

In [17]:
# THE ADVANCE SECTION NODE

def advance_section_node(state: DocSummariserState) -> dict:
    next_index = state["section_index"] + 1
    print(f"\n[NODE: advance_section] Moving to section {next_index+1}/{len(state['sections'])}")

    next_topic = (
        state["sections"][next_index] if next_index < len(state["sections"]) else None
    )

    return {
        "section_index": next_index,
        "current_topic": next_topic if next_topic else state["current_topic"],
        "research_packets": [],       # reset per-section working state
        "researcher_attempts": 0,
        "writer_attempts": 0,
        "follow_up_query": None,
        "needs_more_research": False,
    }

In [18]:
# COMPILE NODE

COMPILE_SYSTEM_PROMPT = """You are a senior financial editor.
You will receive report sections covering different aspects of an Apple FY2024 10-K analysis.

1. Write a 3-sentence Executive Summary at the top
2. Assemble all sections in a logical order beneath it
3. Add a Sources note: "All data sourced from Apple Inc. Form 10-K, Fiscal Year 2024 (ended September 28, 2024)."

Output the complete, publication-ready report.
"""

def compile_node(state: DocSummariserState) -> dict:
    print(f"\n[NODE: compile] Assembling {len(state['report_sections'])} sections")
    sections_text = "\n\n".join(state["report_sections"])

    final_report = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"Report Sections to Compile:\n\n{sections_text}"
        }],
        system=COMPILE_SYSTEM_PROMPT
    )
    return {"final_report": final_report}

In [19]:
# CONDITIONAL EDGES

def route_after_writer(
    state: DocSummariserState
) -> Literal["researcher_node", "advance_section_node", "compile_node"]:
    needs_more = state.get("needs_more_research", False)
    attempts = state.get("researcher_attempts", 0)
    max_attempts = state.get("max_researcher_attempts", 2)

    if needs_more and attempts < max_attempts and state.get("follow_up_query"):
        print(f"  → Edge: needs more research → researcher_node")
        return "researcher_node"

    is_last_section = state["section_index"] >= len(state["sections"]) - 1
    if is_last_section:
        print(f"  → Edge: last section done → compile_node")
        return "compile_node"

    print(f"  → Edge: section done → advance_section_node")
    return "advance_section_node"

In [20]:
# BUILDING THE PRINT GRAPH

def build_summariser_graph(db_path: str = DB_PATH):
    conn = sqlite3.connect(db_path, check_same_thread=False)
    checkpointer = SqliteSaver(conn)

    graph = StateGraph(DocSummariserState)

    graph.add_node("researcher_node", researcher_node)
    graph.add_node("writer_node", writer_node)
    graph.add_node("advance_section_node", advance_section_node)
    graph.add_node("compile_node", compile_node)

    graph.set_entry_point("researcher_node")

    graph.add_edge("researcher_node", "writer_node")
    graph.add_edge("advance_section_node", "researcher_node")
    graph.add_edge("compile_node", END)

    graph.add_conditional_edges(
        "writer_node",
        route_after_writer,
        {
            "researcher_node": "researcher_node",
            "advance_section_node": "advance_section_node",
            "compile_node": "compile_node",
        }
    )

    return graph.compile(checkpointer=checkpointer)


summariser_graph = build_summariser_graph()
print("✅ Document Summariser graph compiled")

try:
    print(summariser_graph.get_graph().draw_ascii())
except Exception:
    print("ASCII visualisation unavailable")

✅ Document Summariser graph compiled
ASCII visualisation unavailable


In [21]:
# THE RUNNER

def run_document_summariser(sections: list[str], max_researcher_attempts: int = 2):
    global usage_tracker
    usage_tracker = TokenUsage()

    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 100}

    print(f"\n{'='*60}")
    print(f"DOCUMENT SUMMARISER — {len(sections)} sections")
    print(f"Thread ID: {thread_id}")
    print(f"{'='*60}")

    initial_state: DocSummariserState = {
        "sections": sections,
        "section_index": 0,
        "current_topic": sections[0],
        "research_packets": [],
        "researcher_attempts": 0,
        "max_researcher_attempts": max_researcher_attempts,
        "follow_up_query": None,
        "needs_more_research": False,
        "writer_attempts": 0,
        "report_sections": [],
        "api_error": False,
        "api_error_type": None,
        "using_fallback_llm": False,
        "error_log": [],
        "final_report": None,
        "finished_at": None,
    }

    final_state = summariser_graph.invoke(initial_state, config=config)
    final_state["finished_at"] = datetime.now().isoformat()
    usage_tracker.report()

    print(f"\n{'='*60}")
    print(f"✅ FINAL DOCUMENT SUMMARY")
    print(f"{'='*60}")
    print(final_state["final_report"])
    print(f"\nSections researched : {len(sections)}")
    print(f"Fallback LLM used    : {final_state['using_fallback_llm']}")
    print(f"Errors encountered   : {len(final_state['error_log'])}")
    print(f"{'='*60}\n")

    filename = f"traces/day20_sprint_{thread_id[:8]}_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(dict(final_state), f, indent=2)
    print(f"📁 Saved to {filename}")

    return final_state

In [22]:
# RUNNING THE FULL SPRINT - Apple 10-K FY2024, 5Sections

sections = [
    "Apple's business overview and primary product/service segments in FY2024",
    "Key risk factors disclosed by Apple in the FY2024 10-K",
    "Apple's financial performance in FY2024 including net sales, net income, and gross margin",
    "Apple's liquidity, capital resources, and cash position at the end of FY2024",
    "Apple's approach to research and development investment in FY2024",
]

final_state = run_document_summariser(sections)


DOCUMENT SUMMARISER — 5 sections
Thread ID: 975fadaa-f3e5-4854-8878-96fde2ea910e

[NODE: researcher] Section 1/5 — Attempt 1
  → Query: Apple's business overview and primary product/service segments in FY2024
  → Coverage: COMPLETE — The three facts together capture Apple’s overall business size and its primary product and service segments for FY 2024.

[NODE: writer] Section 1/5 — Attempt 1
  → Section written (1557 chars) | needs_more: True
  → Edge: needs more research → researcher_node

[NODE: researcher] Section 1/5 — Attempt 2
  → Query: Please provide the specific Apple FY2024 10‑K data on revenue breakdown by produ
  → Coverage: COMPLETE — The three evidences together provide the full FY2024 revenue (net sales) breakdown by both product/service categories and geographic regions as disclosed in Apple’s 10‑K.

[NODE: writer] Section 1/5 — Attempt 2
  → Section written (1596 chars) | needs_more: True
  → Edge: section done → advance_section_node

[NODE: advance_section] Moving to